# Quick test to see how TabPFN performs on this data

In [17]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from tabpfn import TabPFNClassifier
from utils import score_with_thresh
from sklearn.ensemble import RandomForestClassifier
SEED = 3105
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

# numbers of features to test
K_VALUES = [3, 4, 5, 6, 7]
N_RANDOM=3

In [18]:
data_dir = Path("../../data")
X = pd.read_csv(data_dir / "x_train.txt", sep=" ")
y = pd.read_csv(data_dir / "y_train.txt", sep=" ").values.ravel()

with open("../feature_selection/selected_features.txt") as f:
    selected = [s.strip().strip("'").strip('"') for s in f.read().split(",")]

X = X[selected]
# X = X[['V255', 'V191', 'V176']]
print(f"X: {X.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

X: (5000, 30)


In [19]:
rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
ranking = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(
    ascending=False
)
top15 = ranking.index[:15].tolist()


def random_subsets(k, n, exclude):
    seen = {frozenset(exclude)}
    subsets = []
    while len(subsets) < n:
        s = frozenset(rng.choice(top15, size=k, replace=False))
        if s not in seen:
            seen.add(s)
            subsets.append(sorted(s, key=top15.index))
    return subsets


subsets = {}
for k in K_VALUES:
    topk = ranking.index[:k].tolist()
    topk_1 = ranking.index[1 : k + 1].tolist()
    topk_2 = ranking.index[2 : k + 2].tolist()
    subsets[k] = (
        [("top", topk)]
        + [("top+1", topk_1)]
        + [("top+2", topk_2)]
        + [(f"rand_{i}", s) for i, s in enumerate(random_subsets(k, N_RANDOM, topk))]
    )
    print(f"k={k}: {len(subsets[k])} subsets")

k=3: 6 subsets
k=4: 6 subsets
k=5: 6 subsets
k=6: 6 subsets
k=7: 6 subsets


In [3]:
clf = TabPFNClassifier(
    device="cpu", 
    ignore_pretraining_limits=True,
    n_estimators=10, random_state=3105
)

clf.fit(X_train, y_train)
pred = clf.predict_proba(X_test)

In [4]:
score_with_thresh(y_test, pred[:,1], n_var=3, thresh=0)

(np.int64(560), 200)

In [15]:
subsets

{3: [('top', ['V255', 'V176', 'V191']),
  ('top+1', ['V176', 'V191', 'V380']),
  ('top+2', ['V191', 'V380', 'V160']),
  ('rand_0', [np.str_('V160'), np.str_('V416'), np.str_('V224')]),
  ('rand_1', [np.str_('V160'), np.str_('V212'), np.str_('V55')]),
  ('rand_2', [np.str_('V191'), np.str_('V416'), np.str_('V390')])],
 4: [('top', ['V255', 'V176', 'V191', 'V380']),
  ('top+1', ['V176', 'V191', 'V380', 'V160']),
  ('top+2', ['V191', 'V380', 'V160', 'V416']),
  ('rand_0',
   [np.str_('V176'), np.str_('V390'), np.str_('V224'), np.str_('V377')]),
  ('rand_1',
   [np.str_('V380'), np.str_('V416'), np.str_('V224'), np.str_('V212')]),
  ('rand_2',
   [np.str_('V176'), np.str_('V160'), np.str_('V309'), np.str_('V55')])],
 5: [('top', ['V255', 'V176', 'V191', 'V380', 'V160']),
  ('top+1', ['V176', 'V191', 'V380', 'V160', 'V416']),
  ('top+2', ['V191', 'V380', 'V160', 'V416', 'V390']),
  ('rand_0',
   [np.str_('V390'),
    np.str_('V175'),
    np.str_('V262'),
    np.str_('V377'),
    np.str_('V5

In [23]:
best_thresholds = []
best_scores = []

for k in K_VALUES:
    for subset_id, (kind, features) in enumerate(subsets[k]):
        X_train_tmp = X_train[features]

        clf = TabPFNClassifier(
            device="cpu", 
            ignore_pretraining_limits=True,
            n_estimators=10, random_state=3105
        )

        clf.fit(X_train_tmp, y_train)
        proba = clf.predict_proba(X_test[features])[:,1]
        score, contacted = score_with_thresh(y_test, proba, n_var=k, thresh=0)
        print(subset_id, score, contacted)
        # ts = np.linspace(0, 0.7, num=14)
        # best_thresh = 0
        # best_score = -np.inf
        # for t in ts:
        #     score, contacted = score_with_thresh(y_test, proba, n_var=k, thresh=t)
        #     if score >= best_score:
        #         best_score = score
        #         best_thresh = t
        # best_thresholds.append(best_thresh)
        # best_scores.append(best_score)

0 560 200
1 515 200
2 560 200
3 335 200
4 185 200
5 560 200
0 450 200
1 300 200
2 375 200
3 60 200
4 210 200
5 195 200
0 295 200
1 115 200
2 160 200
3 -170 200
4 40 200
5 -35 200
0 95 200
1 -10 200
2 -100 200
3 20 200
4 -10 200
5 -115 200
0 -30 200
1 -165 200
2 -240 200
3 -225 200
4 -105 200
5 -240 200
